In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

customers = [
    (101, "Arun", "Chennai", 30, "2026-08-17 09:00:00"),
    (102, "Kumar", "Bangalore", 35, "2026-08-17 09:05:00"),
    (101, "Arun", "Bangalore", 31, "2026-08-17 10:00:00"),
    (103, "Priya", "Chennai", 27, "2026-08-17 10:10:00"),
    (101, "Arun", "Hyderabad", 32, "2026-08-17 11:00:00")
]

columns = [
    "CustomerId",
    "CustomerName",
    "City",
    "Age",
    "UpdatedAt"
]

df = spark.createDataFrame(customers, columns)

display(df)

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-17 09:00:00
102,Kumar,Bangalore,35,2026-08-17 09:05:00
101,Arun,Bangalore,31,2026-08-17 10:00:00
103,Priya,Chennai,27,2026-08-17 10:10:00
101,Arun,Hyderabad,32,2026-08-17 11:00:00


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze.day8_customer_updates")

In [0]:
%sql
SELECT *
FROM bronze.day8_customer_updates
ORDER BY CustomerId, UpdatedAt;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-17 09:00:00
101,Arun,Bangalore,31,2026-08-17 10:00:00
101,Arun,Hyderabad,32,2026-08-17 11:00:00
102,Kumar,Bangalore,35,2026-08-17 09:05:00
103,Priya,Chennai,27,2026-08-17 10:10:00


In [0]:
%sql
SELECT
    CustomerId,
    COUNT(*) AS RecordCount
FROM bronze.day8_customer_updates
GROUP BY CustomerId
HAVING COUNT(*) > 1;

CustomerId,RecordCount
101,3


In [0]:
duplicate_remove=df.dropDuplicates(['CustomerId'])
display(duplicate_remove)

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Chennai,30,2026-08-17 09:00:00
102,Kumar,Bangalore,35,2026-08-17 09:05:00
103,Priya,Chennai,27,2026-08-17 10:10:00


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec=(Window.partitionBy('CustomerId').orderBy(col('UpdatedAt').desc()))

ranked_df=df.withColumn("row_num",row_number().over(window_spec))

display(ranked_df)

CustomerId,CustomerName,City,Age,UpdatedAt,row_num
101,Arun,Hyderabad,32,2026-08-17 11:00:00,1
101,Arun,Bangalore,31,2026-08-17 10:00:00,2
101,Arun,Chennai,30,2026-08-17 09:00:00,3
102,Kumar,Bangalore,35,2026-08-17 09:05:00,1
103,Priya,Chennai,27,2026-08-17 10:10:00,1


In [0]:
latest_df=(ranked_df.filter(col("row_num")==1).drop("row_num"))
display(latest_df)

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Hyderabad,32,2026-08-17 11:00:00
102,Kumar,Bangalore,35,2026-08-17 09:05:00
103,Priya,Chennai,27,2026-08-17 10:10:00


In [0]:
latest_df.write.format("delta").mode("overwrite").saveAsTable("silver.day8_customers");

In [0]:
%sql
SELECT *
FROM silver.day8_customers
ORDER BY CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Hyderabad,32,2026-08-17 11:00:00
102,Kumar,Bangalore,35,2026-08-17 09:05:00
103,Priya,Chennai,27,2026-08-17 10:10:00


In [0]:
customers = [
    (101, "Arun", "Chennai", 30, "2026-08-17 09:00:00", 1),
    (102, "Kumar", "Bangalore", 35, "2026-08-17 09:05:00", 1),
    (101, "Arun", "Bangalore", 31, "2026-08-17 10:00:00", 2),
    (103, "Priya", "Chennai", 27, "2026-08-17 10:10:00", 2),
    (101, "Arun", "Hyderabad", 32, "2026-08-17 11:00:00", 3),
    (101, "Arun", "Ireland", 32, "2026-08-17 11:00:00", 4)
]

columns = [
    "CustomerId",
    "CustomerName",
    "City",
    "Age",
    "UpdatedAt",
    "BatchId"
]

batch_df = spark.createDataFrame(customers, columns)

display(batch_df)

CustomerId,CustomerName,City,Age,UpdatedAt,BatchId
101,Arun,Chennai,30,2026-08-17 09:00:00,1
102,Kumar,Bangalore,35,2026-08-17 09:05:00,1
101,Arun,Bangalore,31,2026-08-17 10:00:00,2
103,Priya,Chennai,27,2026-08-17 10:10:00,2
101,Arun,Hyderabad,32,2026-08-17 11:00:00,3
101,Arun,Ireland,32,2026-08-17 11:00:00,4


In [0]:
window_batch_spec=(
    Window.partitionBy('CustomerId').
    orderBy(col('UpdatedAt').desc(),
            col('BatchId').desc())
)

ranked_batch_df=batch_df.withColumn("row_num",row_number().over(window_batch_spec))

display(ranked_batch_df)

CustomerId,CustomerName,City,Age,UpdatedAt,BatchId,row_num
101,Arun,Ireland,32,2026-08-17 11:00:00,4,1
101,Arun,Hyderabad,32,2026-08-17 11:00:00,3,2
101,Arun,Bangalore,31,2026-08-17 10:00:00,2,3
101,Arun,Chennai,30,2026-08-17 09:00:00,1,4
102,Kumar,Bangalore,35,2026-08-17 09:05:00,1,1
103,Priya,Chennai,27,2026-08-17 10:10:00,2,1


In [0]:
filterd_batch_df=(ranked_batch_df.filter(col("row_num")==1).drop("row_num"))
display(filterd_batch_df)

CustomerId,CustomerName,City,Age,UpdatedAt,BatchId
101,Arun,Ireland,32,2026-08-17 11:00:00,4
102,Kumar,Bangalore,35,2026-08-17 09:05:00,1
103,Priya,Chennai,27,2026-08-17 10:10:00,2
